# Sensibilidade a três variáveis

A Discussão do artigo aponta três features com problemas conhecidos:
`taxa_trafico`, que mede também a atuação da polícia, e não apenas a
ocorrência do crime; `pib_percapita`, inflado em municípios com polos
industriais; e `i_planejamento_ord`, nota do IEGM em que 70% dos municípios
têm o mesmo valor. Este notebook verifica se os perfis mudam quando cada uma
delas é retirada da modelagem, separadamente e as três ao mesmo tempo.

Cada teste refaz a matriz sem a variável (o peso `1/√n` do bloco é
recalculado com uma coluna a menos), agrupa os municípios com o modelo
registrado no `config.py` e compara a partição obtida com a do
`perfis.csv` pelo ARI (índice de Rand ajustado), como no notebook 05.

Saída: `data/processed/tabela_sensibilidade.csv`.

### Critério de leitura do ARI

Faixas definidas antes da execução e confirmadas pelo grupo em 28/10/2026
(`FAIXAS_ARI`, no `config.py`). Valem também para o notebook 09.

| ARI | Leitura |
|---|---|
| ≥ 0,80 | os perfis se mantêm |
| 0,65 a 0,80 | os perfis mudam em parte; descrever quais |
| < 0,65 | os perfis dependem da mudança testada |

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import adjusted_rand_score

# Os notebooks ficam em notebooks/ e o código do projeto em src/. Estas duas
# linhas apontam o Python para src/, para os "from config import ..." abaixo
# funcionarem tanto rodando daqui quanto da raiz do projeto.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

from config import (ALGORITMO_ESCOLHIDO, BASE_FINAL_CSV, DATA_PROCESSED,
                    DICIONARIO_CSV, K_ESCOLHIDO, VARIAVEIS_SENSIBILIDADE)
from estilo import aplicar_estilo
from preprocessamento import (ORDEM_BLOCOS, carregar_features, montar_matriz,
                              orcamento)
from clusterizacao import faixa_ari, refazer_particao

aplicar_estilo()   # mesma fonte, cores e grade em todas as figuras do artigo
# Se algum import acima falhar, quase sempre é o editor apontando para outra
# instalação do Python. O caminho impresso aqui é o que precisa ter as
# bibliotecas do requirements.txt.
print("Python:", sys.executable)
print(f"Modelo escolhido: {ALGORITMO_ESCOLHIDO}, k = {K_ESCOLHIDO}")

# O código do IBGE é identificador, não quantidade: lido como texto para não
# virar número em nenhuma etapa.
base = pd.read_csv(BASE_FINAL_CSV, dtype={"codigo_ibge": str})
dic = pd.read_csv(DICIONARIO_CSV)
matriz = pd.read_csv(DATA_PROCESSED / "matriz_modelagem.csv",
                     dtype={"codigo_ibge": str})
perfis = pd.read_csv(DATA_PROCESSED / "perfis.csv", dtype={"codigo_ibge": str})

Python: D:\RP 2\database-rp2\.venv\Scripts\python.exe
Modelo escolhido: kmeans, k = 4


## 1. Partição de referência

A comparação é feita contra os perfis do `perfis.csv`, gerados no notebook
05. A célula abaixo confere que a matriz montada pela função de
`src/preprocessamento.py` é a mesma do `matriz_modelagem.csv` e que a função
`refazer_particao`, usada nos testes, reproduz exatamente esses perfis
quando nenhuma variável é retirada.

In [2]:
features, bloco_de = carregar_features(dic)
# Mesmo filtro e mesma ordem do notebook 05 (sem a capital, que não tem IEGM).
modelagem = base[~base["flag_sem_iegm"]].reset_index(drop=True)
assert (modelagem["codigo_ibge"] == perfis["codigo_ibge"]).all(), \
    "perfis.csv fora da ordem da base"
referencia = perfis["perfil"].to_numpy()

_, W, _ = montar_matriz(modelagem, features, bloco_de)
assert np.allclose(W, matriz[features]), "W diferente do matriz_modelagem.csv"
# Com as 22 features, a função tem que devolver os mesmos perfis, com a
# mesma numeração; senão a comparação abaixo não valeria nada.
perfil_22, _, _ = refazer_particao(modelagem, features, bloco_de,
                                   ALGORITMO_ESCOLHIDO, K_ESCOLHIDO)
assert (perfil_22 == referencia).all(), "refazer_particao não reproduz o perfis.csv"
print(f"ok: {len(referencia)} municípios, mesmos perfis do perfis.csv")

ok: 644 municípios, mesmos perfis do perfis.csv


## 2. Retirada das variáveis

Para cada linha do teste (cada variável sozinha e as três juntas), a célula
abaixo retira as colunas, refaz a matriz e a partição e compara o resultado
com a referência. Antes de agrupar, um `assert` confere que cada bloco
continua com um terço da variância total: como o peso é `1/√n`, o bloco que
perde uma coluna passa a ter um peso um pouco maior em cada coluna restante,
de modo a não valer menos na distância.

A coluna `n_colunas_no_bloco` indica quantas colunas sobraram no bloco de
onde a variável saiu, e `municipios_que_mudaram` conta os municípios cujo
perfil, depois da renumeração pelo critério socioeconômico, é diferente do
original.

In [3]:
# Cada item é (nome da linha, colunas retiradas).
retiradas = {v: [v] for v in VARIAVEIS_SENSIBILIDADE}
retiradas["as três"] = VARIAVEIS_SENSIBILIDADE

novos, linhas = {}, []
for nome, retirar in retiradas.items():
    ficam = [c for c in features if c not in retirar]
    perfil_novo, _, W_novo = refazer_particao(modelagem, ficam, bloco_de,
                                              ALGORITMO_ESCOLHIDO, K_ESCOLHIDO)
    # Se o peso não acompanhasse a coluna retirada, o bloco dela valeria
    # menos de um terço e o teste mudaria duas coisas ao mesmo tempo.
    assert np.allclose(orcamento(W_novo, bloco_de), 100 / 3), \
        f"sem {nome}: algum bloco fora de um terço"
    novos[nome] = perfil_novo

    blocos = sorted({bloco_de[c] for c in retirar}, key=ORDEM_BLOCOS.index)
    n_bloco = pd.Series([bloco_de[c] for c in ficam]).value_counts()
    ari = adjusted_rand_score(referencia, perfil_novo)
    linhas.append({
        "variavel_retirada": nome,
        "bloco": ", ".join(blocos),
        "n_colunas_no_bloco": ", ".join(str(n_bloco[b]) for b in blocos),
        "ari": ari,
        "leitura": faixa_ari(ari),
        "municipios_que_mudaram": int((perfil_novo != referencia).sum()),
    })

tabela = pd.DataFrame(linhas)
assert len(tabela) == 4, "deveriam ser 3 variáveis + as três juntas"
tabela.to_csv(DATA_PROCESSED / "tabela_sensibilidade.csv", index=False)
tabela.round(3)

,variavel_retirada,bloco,n_colunas_no_bloco,ari,leitura,municipios_que_mudaram
0,taxa_trafico,criminalidade,8,0.948,os perfis se mantêm,12
1,pib_percapita,socioeconomico,5,0.864,os perfis se mantêm,33
2,i_planejamento_ord,gestao,6,0.904,os perfis se mantêm,23
3,as três,"criminalidade, socioeconomico, gestao","8, 5, 6",0.849,os perfis se mantêm,37


## 3. Tabelas cruzadas

Cada tabela mostra, nas linhas, o perfil original de cada município e, nas
colunas, o perfil obtido sem a variável indicada, numerado pelo mesmo
critério socioeconômico. Os municípios fora da diagonal são os que mudaram de
perfil.

In [4]:
for nome, perfil_novo in novos.items():
    print(f"Sem {nome}:")
    # crosstab conta quantos municípios há em cada combinação de perfis.
    display(pd.crosstab(pd.Series(referencia, name="perfil original"),
                        pd.Series(perfil_novo, name=f"sem {nome}"),
                        margins=True, margins_name="total"))

Sem taxa_trafico:


sem taxa_trafico,1,2,3,4,total
perfil original,,,,,
1,65,1,0,0,66
2,0,215,1,1,217
3,1,4,191,0,196
4,0,1,3,161,165
total,66,221,195,162,644


Sem pib_percapita:


sem pib_percapita,1,2,3,4,total
perfil original,,,,,
1,65,1,0,0,66
2,2,215,0,0,217
3,2,8,186,0,196
4,0,5,15,145,165
total,69,229,201,145,644


Sem i_planejamento_ord:


sem i_planejamento_ord,1,2,3,4,total
perfil original,,,,,
1,66,0,0,0,66
2,2,211,2,2,217
3,3,4,186,3,196
4,0,1,6,158,165
total,71,216,194,163,644


Sem as três:


sem as três,1,2,3,4,total
perfil original,,,,,
1,65,1,0,0,66
2,2,210,4,1,217
3,3,6,187,0,196
4,0,0,20,145,165
total,70,217,211,146,644


## 4. Leitura

Os quatro testes ficam na faixa em que os perfis se mantêm (ARI ≥ 0,80).
Uma frase por linha da tabela, no registro do artigo:

- **`taxa_trafico`.** A retirada da taxa de tráfico de entorpecentes produz
  ARI de 0,948 em relação à partição de referência, com 12 dos 644
  municípios mudando de perfil, de modo que os perfis se mantêm.
- **`pib_percapita`.** Sem o PIB per capita, o ARI é de 0,864 e 33
  municípios mudam de perfil, a maior parte deles saindo do perfil 4 (15
  para o perfil 3 e 5 para o perfil 2) e do perfil 3 (8 para o perfil 2);
  os perfis se mantêm.
- **`i_planejamento_ord`.** A retirada da nota de planejamento do IEGM
  produz ARI de 0,904, com 23 municípios mudando de perfil, e os perfis se
  mantêm.
- **As três ao mesmo tempo.** Com as três variáveis retiradas, o ARI é de
  0,849 e 37 municípios mudam de perfil, 20 deles do perfil 4 para o perfil
  3; os perfis se mantêm.

Sem o PIB per capita, sem a nota de planejamento e com as três retiradas, a
mudança mais frequente é a do perfil 4 para o perfil 3 (15, 6 e 20
municípios, respectivamente); sem o tráfico, é a do perfil 3 para o perfil 2
(4 municípios). Em todos os testes, o perfil 1 mantém pelo menos 65 dos seus
66 municípios.